# Post 3 — Interpretability (§9)

LightGBM TreeSHAP only — global importance, top-5 SHAP profiles, and one local waterfall. Per §9, TabPFN-TS interpretability is treated **qualitatively as a deployment constraint**: no native SHAP, no attention extraction attempted. The absence is itself the finding — see the markdown note at the end. Do not spend effort forcing explanations out of TabPFN-TS.

## 0. Control panel

In [ ]:
# ---- Control panel (interpretability §9) ----------------------------------
CONFIG_NAME = "default.yaml"
OUTPUT_SUBDIR = "interpret"
FEATURE_SET = "full"
USE_CORE_SLICE = True
SHAP_SAMPLE_SIZE = 5000
QUICK = True
RANDOM_SEED = 42

## 1. Imports + helpers

In [ ]:
import sys, time, json, copy, platform, subprocess
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or repo root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "demand").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.demand.config import load_config, resolve_path
from src.demand.data.load import load_raw_data, build_panel
from src.demand.data.splits import (
    build_main_splits, build_core_slice, family_regime, regime_horizon, regime_costs,
)
from src.demand.data.supervised_frame import build_supervised_frame
from src.demand.models import lgbm_point, lgbm_quantile
from src.demand.models.lgbm_base import train_lgbm

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "no-git"

In [ ]:
def point_params(cfg):
    p = dict(cfg["lgbm"]["defaults"])
    p.update(cfg["lgbm"]["objectives"]["point"])
    return p

def train_point(panel_, feature_set, train_origins, val_origins=None,
                     series_filter=None, include_earthquake=True):
    """Build the supervised frame from `panel_` and fit one point (log1p+L2) booster."""
    frame_train = build_supervised_frame(
        panel_, origins=train_origins, horizon=cfg["horizon_days"],
        feature_set=feature_set, config=cfg, mode="train",
        holidays=raw.holidays, stores=raw.stores, series_filter=series_filter,
        include_earthquake=include_earthquake,
    )
    frame_val = None
    if val_origins:
        frame_val = build_supervised_frame(
            panel_, origins=val_origins, horizon=cfg["horizon_days"],
            feature_set=feature_set, config=cfg, mode="train",
            holidays=raw.holidays, stores=raw.stores, series_filter=series_filter,
            include_earthquake=include_earthquake,
        )
    params = point_params(cfg)
    n_est = params.pop("n_estimators", 2000)
    if QUICK:
        n_est = min(n_est, 400)
    return train_lgbm(
        frame_train=frame_train, frame_val=frame_val, feature_set=feature_set,
        params=params, n_estimators=n_est,
        early_stopping_rounds=cfg["lgbm"]["search_space"]["early_stopping_rounds"],
        include_earthquake=include_earthquake, name=f"lgbm_point_{feature_set}",
    )

## 2. Load data + scope

In [ ]:
cfg = copy.deepcopy(load_config(CONFIG_NAME))
cfg["seed"] = RANDOM_SEED
splits = build_main_splits(cfg)

t0 = time.time()
raw = load_raw_data(cfg)
panel = build_panel(raw, include_test=False)
print(f"loaded panel in {time.time()-t0:.1f}s | rows={len(panel):,} | "
      f"series={panel[['store_nbr','family']].drop_duplicates().shape[0]:,}")

if USE_CORE_SLICE:
    series_filter = build_core_slice(panel, raw.stores, cfg, write_csv=False)
    print(f"core slice: {len(series_filter)} series, "
          f"{series_filter['family'].nunique()} families")
else:
    series_filter = None
    print("scope: full panel")

results_dir = resolve_path(cfg, "results_dir")
figures_dir = resolve_path(cfg, "figures_dir")
out_dir = results_dir / OUTPUT_SUBDIR
out_dir.mkdir(parents=True, exist_ok=True)
print(f"git commit: {git_commit()}  ->  output: {out_dir}")

## 3. Global TreeSHAP

In [ ]:
from src.demand.interpret import shap_lgbm
from src.demand.data.supervised_frame import split_xy

train_origins = list(splits.training_origins)
art = train_point(panel, FEATURE_SET, train_origins, series_filter=series_filter)

# Build the rows we explain (a forecast frame on the test origins).
frame_fc = build_supervised_frame(
    panel, origins=list(splits.test_origins), horizon=cfg["horizon_days"],
    feature_set=FEATURE_SET, config=cfg, mode="forecast",
    holidays=raw.holidays, stores=raw.stores, series_filter=series_filter)
X, _ = split_xy(frame_fc, feature_set=FEATURE_SET, drop_missing_y=False)

bundle = shap_lgbm.compute_shap(art, X, sample_size=SHAP_SAMPLE_SIZE, seed=RANDOM_SEED)
display(bundle.global_importance.head(15))
shap_lgbm.plot_global_importance(bundle, figures_dir / OUTPUT_SUBDIR)
shap_lgbm.plot_top5_profiles(bundle, figures_dir / OUTPUT_SUBDIR)
bundle.global_importance.to_parquet(results_dir / "shap_global_importance.parquet", index=False)

## 4. Local explanation

In [ ]:
# §9 — one illustrative local explanation: PRODUCE at the largest Quito store.
quito = raw.stores[raw.stores["city"] == "Quito"]["store_nbr"].tolist()
cand = frame_fc[(frame_fc["family"] == "PRODUCE") & (frame_fc["store_nbr"].isin(quito))]
if cand.empty:
    cand = frame_fc[frame_fc["family"] == "PRODUCE"]
if not cand.empty:
    row = cand.iloc[[0]]
    Xrow, _ = split_xy(row, feature_set=FEATURE_SET, drop_missing_y=False)
    path = shap_lgbm.local_explanation(art, Xrow, figures_dir / OUTPUT_SUBDIR,
                                       "local_produce_quito.png")
    print("local explanation ->", path)
else:
    print("no PRODUCE rows in scope; widen USE_CORE_SLICE or feature set")

## 5. TabPFN-TS interpretability — the gap (qualitative)

TabPFN-TS ships no TreeSHAP-equivalent and we do not attempt KernelSHAP (Post 2 showed it is prohibitively slow on TabPFN). For the blog post, document honestly: a planner cannot ask the model *why* a forecast moved. In a domain where planners override models, that opacity is a real production cost, independent of accuracy.